# Debug Session

The `DebugSession` API provides an interactive debugging workflow for conda recipes.
It sets up the full build environment (resolves dependencies, fetches sources,
installs environments, creates build script) **without** running the build.
You can then iteratively run and re-run the build script, inspecting output each time.

This is ideal for:
- Debugging build failures interactively
- AI agents that need to fix and retry builds programmatically
- Understanding what happens during a build step by step

## Step 1: Parse and Render a Recipe

First, create a simple recipe and render it with variants:

In [ ]:
from rattler_build import DebugSession, Stage0Recipe, VariantConfig, RenderConfig

recipe_yaml = """
package:
  name: debug-demo
  version: "1.0.0"

build:
  number: 0
  noarch: generic
  script:
    content: |
      echo "Hello from the build script!"
      mkdir -p $PREFIX/bin
      echo "hello from debug-demo" > $PREFIX/bin/hello

requirements:
  run:
    - python
"""

recipe = Stage0Recipe.from_yaml(recipe_yaml)
variants = recipe.render(VariantConfig(), RenderConfig())
print(f"Rendered {len(variants)} variant(s)")

## Step 2: Set Up the Debug Environment

`DebugSession.setup()` resolves dependencies, fetches sources, installs environments,
and creates the build script — all without running the actual build.

In [ ]:
session = DebugSession.setup(
    rendered_variant=variants[0],
    channels=["conda-forge"],
)

print(f"Work directory:  {session.work_dir}")
print(f"Host prefix:     {session.host_prefix}")
print(f"Build prefix:    {session.build_prefix}")
print(f"Build script:    {session.build_script}")
print(f"Output dir:      {session.output_dir}")
print(f"Setup log:       {len(session.log)} messages")

## Step 3: Inspect the Environment

Before running the build, inspect the generated files:

In [ ]:
# Show build script contents
print("Build script contents:")
print(session.build_script.read_text())

# List files in work directory
print("\nWork directory contents:")
for p in sorted(session.work_dir.iterdir()):
    print(f"  {p.name}")

## Step 4: Run the Build Script

Run the build script with `trace=True` to see each command as it executes:

In [ ]:
result = session.run(trace=True)

print(f"Exit code: {result.exit_code}")
print(f"\nStdout:\n{result.stdout}")
if result.stderr:
    print(f"\nStderr:\n{result.stderr}")

## Step 5: Iterate — Fix and Re-run

The key advantage: modify files and re-run without repeating the expensive setup.

In [ ]:
# Modify the build script
content = session.build_script.read_text()
content = content.replace("echo hello", "echo 'hello world'")
session.build_script.write_text(content)

# Re-run — fast since environments are already installed
result = session.run(trace=True)
print(f"Exit code: {result.exit_code}")
print(f"Stdout:\n{result.stdout}")

## Step 6: Programmatic Error Handling

For automated workflows, check `exit_code` and parse stderr:

In [ ]:
result = session.run()

if result.exit_code != 0:
    print("Build failed!")
    print(f"stderr: {result.stderr}")
    if "command not found" in result.stderr:
        print("Missing command - may need a build dependency")
    elif "No such file or directory" in result.stderr:
        print("Missing file - check source extraction")
else:
    print("Build succeeded!")

## Summary

The `DebugSession` workflow:

1. **`DebugSession.setup()`** — Resolves deps, fetches sources, installs envs, creates build script
2. **Inspect** — Check `work_dir`, `host_prefix`, `build_prefix`, `build_script`
3. **`session.run()`** — Run the build script, get `exit_code`, `stdout`, `stderr`
4. **Iterate** — Modify files, call `run()` again (no re-setup needed)

This enables fast iteration loops for both human debugging and AI-driven build automation.